In [ ]:
!pip install -q wfdb neurokit2 scikit-learn imbalanced-learn xgboost lightgbm catboost
!pip install -q seaborn plotly shap torchmetrics torchsummary
!pip install -q kan  # Kolmogorov-Arnold Networks — SOTA 2025 for CTG
!pip install -q scipy statsmodels

In [ ]:
import os, warnings, time, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import wfdb
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis, f_oneway
import neurokit2 as nk
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              balanced_accuracy_score, cohen_kappa_score,
                              ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchmetrics import F1Score, AUROC, Accuracy
import torchsummary

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
os.makedirs('/content/fetal', exist_ok=True)

# --- Dataset 1: UCI CTG (tabular, 2126 records) ---
print("Downloading UCI CTG dataset...")
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/00193/CTG.xls" \
     -O /content/fetal/CTG.xls
print("UCI CTG downloaded.")

# --- Dataset 2: CTU-UHB raw waveforms (PhysioNet, no credentialing needed) ---
print("\nDownloading CTU-UHB waveform database (552 recordings @ 4 Hz)...")
!wget -q -r -N -c -np \
     "https://physionet.org/files/ctu-uhb-ctgdb/1.0.0/" \
     -P /content/fetal/ctu_uhb/ \
     --reject "index.html*"
print("CTU-UHB downloaded.")

import glob
ctu_files = glob.glob('/content/fetal/ctu_uhb/**/*.hea', recursive=True)
print(f"CTU-UHB recordings found: {len(ctu_files)}")


In [ ]:
# UCI CTG has 3 sheets — use the correct one
df_raw = pd.read_excel('/content/fetal/CTG.xls', sheet_name='Raw Data', header=1)

# Keep only the 21 features + NSP (Normal/Suspect/Pathological) label
feature_cols = [
    'LB',    # FHR baseline (bpm)
    'AC',    # Accelerations/sec
    'FM',    # Fetal movements/sec
    'UC',    # Uterine contractions/sec
    'DL',    # Light decelerations/sec
    'DS',    # Severe decelerations/sec
    'DP',    # Prolonged decelerations/sec
    'ASTV',  # % time with abnormal short-term variability
    'MSTV',  # Mean short-term variability (ms)
    'ALTV',  # % time with abnormal long-term variability
    'MLTV',  # Mean long-term variability (ms)
    'Width', # Histogram width
    'Min',   # Histogram min
    'Max',   # Histogram max
    'Nmax',  # Number of histogram peaks
    'Nzeros',# Number of zeros in histogram
    'Mode',  # Histogram mode
    'Mean',  # Histogram mean
    'Median',# Histogram median
    'Variance',# Histogram variance
    'Tendency' # Histogram tendency
]
label_col = 'NSP'  # 1=Normal, 2=Suspect, 3=Pathological

# Validate columns exist
available = [c for c in feature_cols + [label_col] if c in df_raw.columns]
df = df_raw[available].dropna().copy()
df[label_col] = df[label_col].astype(int)
df = df[df[label_col].isin([1, 2, 3])]  # remove any noise rows

# Remap labels: 0=Normal, 1=Suspect, 2=Pathological
df['label'] = df[label_col] - 1
label_names = {0: 'Normal', 1: 'Suspect', 2: 'Pathological'}

print(f"UCI CTG shape: {df.shape}")
print(f"\nClass distribution:")
for k, v in label_names.items():
    count = (df['label'] == k).sum()
    pct = count / len(df) * 100
    print(f"  {v:15s}: {count:4d} ({pct:.1f}%)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class counts
class_counts = df['label'].value_counts().sort_index()
colors_cls = ['#2ecc71', '#f39c12', '#e74c3c']
bars = axes[0].bar([label_names[i] for i in class_counts.index],
                    class_counts.values,
                    color=colors_cls, edgecolor='black', alpha=0.85)
axes[0].set_title('Class Distribution (UCI CTG)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Imbalance ratio
imbalance = class_counts.max() / class_counts.min()
axes[0].set_xlabel(f'Imbalance ratio: {imbalance:.1f}x', fontsize=10)

# FHR Baseline by class
for i, (cls_id, cls_name) in enumerate(label_names.items()):
    mask = df['label'] == cls_id
    axes[1].hist(df.loc[mask, 'LB'], bins=30, alpha=0.6,
                 label=cls_name, color=colors_cls[i], density=True)
axes[1].set_title('FHR Baseline Distribution by Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('FHR Baseline (bpm)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].axvline(110, color='red', linestyle='--', linewidth=1, alpha=0.7, label='110 bpm (lower normal)')
axes[1].axvline(160, color='red', linestyle='--', linewidth=1, alpha=0.7, label='160 bpm (upper normal)')
axes[1].grid(True, alpha=0.3)

# Variability by class
for i, (cls_id, cls_name) in enumerate(label_names.items()):
    mask = df['label'] == cls_id
    axes[2].scatter(df.loc[mask, 'MSTV'], df.loc[mask, 'MLTV'],
                    alpha=0.3, s=15, label=cls_name, color=colors_cls[i])
axes[2].set_title('Short vs Long-term Variability', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Mean Short-term Variability (ms)')
axes[2].set_ylabel('Mean Long-term Variability (ms)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('UCI CTG: Exploratory Data Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_ctg_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
features_to_plot = ['LB', 'AC', 'FM', 'UC', 'ASTV', 'MSTV', 'ALTV', 'MLTV',
                    'DL', 'DS', 'DP', 'Variance']

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

for idx, feat in enumerate(features_to_plot):
    if feat not in df.columns:
        continue
    ax = axes[idx]
    groups = [df[df['label'] == k][feat].dropna().values for k in [0, 1, 2]]

    # ANOVA test
    f_stat, p_val = f_oneway(*groups)

    for i, (group, cls_name) in enumerate(zip(groups, label_names.values())):
        ax.hist(group, bins=25, alpha=0.55, label=cls_name,
                color=colors_cls[i], density=True)

    ax.set_title(f'{feat}\n(ANOVA p={p_val:.2e})', fontsize=9,
                 fontweight='bold',
                 color='#e74c3c' if p_val < 0.001 else '#f39c12')
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distributions by Class (ANOVA significance)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Correlation heatmap
feat_df = df[[c for c in feature_cols if c in df.columns]]
corr = feat_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            ax=axes[0], annot_kws={'size': 7}, center=0,
            linewidths=0.3, vmin=-1, vmax=1)
axes[0].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)

# Dawes-Redman criteria mapping
# Key criteria: Baseline 110-160, STV>3ms, Accelerations>0, No decels
dr_features = ['LB', 'MSTV', 'AC', 'DL', 'DS', 'DP', 'ASTV']
dr_threshold = {
    'LB':   (110, 160),   # Normal range
    'MSTV': (3, 100),     # STV > 3ms
    'AC':   (0.001, 1),   # At least some accelerations
    'DL':   (0, 0.003),   # Minimal light decels
    'DS':   (0, 0.001),   # No severe decels
    'DP':   (0, 0.001),   # No prolonged decels
    'ASTV': (0, 20),      # <20% abnormal STV
}

class_means = df.groupby('label')[dr_features].mean()

x = np.arange(len(dr_features))
width = 0.25
for i, (cls_id, cls_name) in enumerate(label_names.items()):
    axes[1].bar(x + i*width, class_means.loc[cls_id],
                width, label=cls_name, color=colors_cls[i],
                alpha=0.85, edgecolor='black')

axes[1].set_title('Mean Feature Values by Class\n(Dawes-Redman Key Features)',
                   fontsize=13, fontweight='bold')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(dr_features, rotation=20, fontsize=9)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('eda_correlations_dr.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import glob

CTU_PATH = '/content/fetal/ctu_uhb/physionet.org/files/ctu-uhb-ctgdb/1.0.0'

def load_ctu_record(record_path):
    """Load FHR + UC waveform and clinical metadata from CTU-UHB."""
    try:
        record = wfdb.rdrecord(record_path)
        signals = record.p_signal  # (N, 2): FHR, UC — sampled at 4 Hz
        fhr = signals[:, 0]
        uc  = signals[:, 1]

        # Load clinical annotations
        header = wfdb.rdheader(record_path)
        comments = {c.split(':')[0].strip(): c.split(':')[1].strip()
                    for c in header.comments if ':' in c}

        # Key outcome: pH < 7.15 = acidosis (pathological)
        ph = float(comments.get('pH', '7.2'))
        label = 0 if ph >= 7.15 else 1  # binary: normal vs acidosis

        return {'fhr': fhr, 'uc': uc, 'ph': ph, 'label': label,
                'comments': comments}
    except Exception as e:
        return None

# Load all records
record_paths = [p.replace('.hea', '') for p in
                glob.glob(f'{CTU_PATH}/*.hea')]
print(f"Loading {len(record_paths)} CTU-UHB recordings...")

ctu_records = []
for rp in record_paths:
    rec = load_ctu_record(rp)
    if rec is not None:
        ctu_records.append(rec)

print(f"Successfully loaded: {len(ctu_records)} recordings")
normal_count = sum(1 for r in ctu_records if r['label'] == 0)
acidosis_count = sum(1 for r in ctu_records if r['label'] == 1)
print(f"  Normal (pH≥7.15): {normal_count}")
print(f"  Acidosis (pH<7.15): {acidosis_count}")


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
t_scale = np.arange(0, 4*60, 1/4) / 60  # minutes

# Sample normal vs acidosis FHR waveforms
normal_samples = [r for r in ctu_records if r['label'] == 0][:3]
acidosis_samples = [r for r in ctu_records if r['label'] == 1][:3]

for row_idx, (samples, title, color) in enumerate([
    (normal_samples[:1], 'Normal (pH≥7.15)', '#2ecc71'),
    (acidosis_samples[:1], 'Acidosis (pH<7.15)', '#e74c3c'),
]):
    if not samples:
        continue
    rec = samples[0]
    fhr = rec['fhr'][:4*60*4]  # First 4 minutes
    uc  = rec['uc'][:4*60*4]
    t   = np.arange(len(fhr)) / 4 / 60

    axes[row_idx, 0].plot(t, fhr, color=color, linewidth=0.8)
    axes[row_idx, 0].axhline(110, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    axes[row_idx, 0].axhline(160, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    axes[row_idx, 0].fill_between(t, 110, 160, alpha=0.1, color='green')
    axes[row_idx, 0].set_title(f'FHR — {title}\npH={rec["ph"]:.3f}',
                                fontweight='bold', color=color)
    axes[row_idx, 0].set_xlabel('Time (minutes)')
    axes[row_idx, 0].set_ylabel('FHR (bpm)')
    axes[row_idx, 0].set_ylim(60, 200)
    axes[row_idx, 0].grid(True, alpha=0.3)

    axes[row_idx, 1].plot(t, uc, color='#3498db', linewidth=0.8)
    axes[row_idx, 1].set_title(f'Uterine Contractions — {title}', fontweight='bold')
    axes[row_idx, 1].set_xlabel('Time (minutes)')
    axes[row_idx, 1].set_ylabel('UC Signal')
    axes[row_idx, 1].grid(True, alpha=0.3)

# pH distribution
ph_values = [r['ph'] for r in ctu_records]
axes[2, 0].hist([r['ph'] for r in ctu_records if r['label'] == 0], bins=30,
                alpha=0.7, label='Normal', color='#2ecc71', density=True)
axes[2, 0].hist([r['ph'] for r in ctu_records if r['label'] == 1], bins=30,
                alpha=0.7, label='Acidosis', color='#e74c3c', density=True)
axes[2, 0].axvline(7.15, color='black', linestyle='--', linewidth=2,
                   label='pH 7.15 threshold')
axes[2, 0].set_title('Umbilical Artery pH Distribution', fontweight='bold')
axes[2, 0].set_xlabel('pH')
axes[2, 0].set_ylabel('Density')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# FHR baseline distribution
normal_baselines = [np.median(r['fhr'][r['fhr'] > 0]) for r in ctu_records if r['label'] == 0]
acid_baselines   = [np.median(r['fhr'][r['fhr'] > 0]) for r in ctu_records if r['label'] == 1]
axes[2, 1].hist(normal_baselines,  bins=25, alpha=0.7, color='#2ecc71', label='Normal', density=True)
axes[2, 1].hist(acid_baselines,    bins=25, alpha=0.7, color='#e74c3c', label='Acidosis', density=True)
axes[2, 1].axvline(110, color='gray', linestyle='--', linewidth=1)
axes[2, 1].axvline(160, color='gray', linestyle='--', linewidth=1)
axes[2, 1].set_title('FHR Median Baseline by Outcome', fontweight='bold')
axes[2, 1].set_xlabel('FHR Baseline (bpm)')
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

plt.suptitle('CTU-UHB Raw Waveform EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_ctu_waveforms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def extract_fhr_features(fhr, uc, fs=4):
    """
    Extract Dawes-Redman + clinical CTG features from raw FHR + UC signals.
    Matches exactly what MedVerse's ctg_dawes_redman.py computes.
    fs: sampling rate (4 Hz for CTU-UHB)
    """
    fhr_clean = fhr[fhr > 50]  # remove artifacts (< 50 bpm invalid)
    if len(fhr_clean) < fs * 60:  # need at least 1 minute
        return None

    features = {}

    # === Baseline FHR ===
    features['baseline_fhr']   = np.percentile(fhr_clean, 50)  # trimmed median
    features['fhr_mean']       = fhr_clean.mean()
    features['fhr_std']        = fhr_clean.std()

    # === Short-term variability (STV) ===
    # FIGO: beat-to-beat variability — successive differences
    rr_intervals = 60000 / fhr_clean  # convert bpm to ms
    successive_diffs = np.abs(np.diff(rr_intervals))
    features['stv_ms']         = successive_diffs.mean()  # STV in ms
    features['rmssd_ms']       = np.sqrt(np.mean(successive_diffs**2))

    # === Long-term variability (LTV) ===
    # Minute-by-minute oscillation
    window = fs * 60  # 1-minute windows
    minute_means = [fhr_clean[i:i+window].mean()
                    for i in range(0, len(fhr_clean)-window, window//2)]
    features['ltv_bpm']        = np.std(minute_means) if minute_means else 0

    # === Accelerations ===
    baseline = features['baseline_fhr']
    accel_threshold = baseline + 15  # FIGO: ≥15 bpm above baseline
    accel_mask = fhr_clean > accel_threshold
    accel_runs = _count_runs(accel_mask, min_duration=fs*15)  # ≥15 seconds
    features['n_accelerations'] = accel_runs
    features['accel_rate']      = accel_runs / (len(fhr_clean) / fs / 60)

    # === Decelerations ===
    decel_threshold = baseline - 15
    decel_mask = fhr_clean < decel_threshold
    # Late decelerations: occur after UC peak
    early_decel_runs = _count_runs(decel_mask, min_duration=fs*10)
    features['n_decelerations'] = early_decel_runs
    features['decel_rate']      = early_decel_runs / (len(fhr_clean) / fs / 60)

    # === Reactivity ===
    features['reactive']        = int(accel_runs >= 2 and features['stv_ms'] >= 3)

    # === FHR Range ===
    features['fhr_range']       = np.percentile(fhr_clean, 95) - np.percentile(fhr_clean, 5)

    # === Power spectral analysis (4 bands) ===
    if len(fhr_clean) > fs * 60 * 2:
        f, psd = scipy_signal.welch(fhr_clean, fs=fs, nperseg=min(256, len(fhr_clean)//4))
        vlf_mask = (f >= 0.0033) & (f < 0.04)
        lf_mask  = (f >= 0.04)  & (f < 0.15)
        mf_mask  = (f >= 0.15)  & (f < 0.5)
        hf_mask  = (f >= 0.5)   & (f < 1.0)
        total_power = psd.sum() + 1e-10
        features['vlf_power']  = psd[vlf_mask].sum() / total_power
        features['lf_power']   = psd[lf_mask].sum() / total_power
        features['mf_power']   = psd[mf_mask].sum() / total_power
        features['hf_power']   = psd[hf_mask].sum() / total_power
        features['lf_hf_ratio']= (psd[lf_mask].sum() / (psd[hf_mask].sum() + 1e-10))
    else:
        for key in ['vlf_power','lf_power','mf_power','hf_power','lf_hf_ratio']:
            features[key] = 0.0

    # === UC features ===
    if uc is not None and len(uc) > 0:
        uc_clean = uc[uc >= 0]
        features['uc_mean']    = uc_clean.mean()
        features['uc_std']     = uc_clean.std()
        uc_peaks, _ = scipy_signal.find_peaks(uc_clean, height=uc_clean.mean()+uc_clean.std(),
                                              distance=fs*60)
        features['n_contractions'] = len(uc_peaks)
        features['contraction_rate'] = len(uc_peaks) / (len(uc_clean) / fs / 60)
    else:
        for key in ['uc_mean','uc_std','n_contractions','contraction_rate']:
            features[key] = 0.0

    return features

def _count_runs(mask, min_duration):
    """Count runs of True in mask with minimum duration."""
    count, in_run, run_len = 0, False, 0
    for v in mask:
        if v:
            in_run, run_len = True, run_len + 1
        elif in_run:
            if run_len >= min_duration:
                count += 1
            in_run, run_len = False, 0
    if in_run and run_len >= min_duration:
        count += 1
    return count

# Extract features from all CTU-UHB records
print("Extracting CTG features from CTU-UHB waveforms...")
ctu_features = []
for rec in ctu_records:
    feats = extract_fhr_features(rec['fhr'], rec['uc'])
    if feats:
        feats['label'] = rec['label']  # 0=normal, 1=acidosis
        feats['ph']    = rec['ph']
        ctu_features.append(feats)

ctu_df = pd.DataFrame(ctu_features).dropna()
print(f"CTU-UHB feature matrix: {ctu_df.shape}")
print(f"Features extracted: {[c for c in ctu_df.columns if c not in ['label','ph']]}")


In [ ]:
# === UCI CTG preprocessing ===
uci_feature_cols_avail = [c for c in feature_cols if c in df.columns]
X_uci = df[uci_feature_cols_avail].values.astype(float)
y_uci = df['label'].values  # 0=Normal, 1=Suspect, 2=Pathological

scaler_uci = StandardScaler()
X_uci_scaled = scaler_uci.fit_transform(X_uci)

# SMOTETomek — oversamples minority AND removes Tomek links (noisy majority)
smote_tomek = SMOTETomek(random_state=42)
X_uci_bal, y_uci_bal = smote_tomek.fit_resample(X_uci_scaled, y_uci)

print(f"UCI CTG before SMOTE: {dict(zip(*np.unique(y_uci, return_counts=True)))}")
print(f"UCI CTG after  SMOTE: {dict(zip(*np.unique(y_uci_bal, return_counts=True)))}")

# === CTU-UHB preprocessing ===
ctu_feat_cols = [c for c in ctu_df.columns if c not in ['label', 'ph']]
X_ctu = ctu_df[ctu_feat_cols].values.astype(float)
y_ctu = ctu_df['label'].values  # 0=normal, 1=acidosis

scaler_ctu = StandardScaler()
X_ctu_scaled = scaler_ctu.fit_transform(X_ctu)

smote = SMOTE(random_state=42, k_neighbors=min(5, y_ctu.sum()-1))
X_ctu_bal, y_ctu_bal = smote.fit_resample(X_ctu_scaled, y_ctu)

print(f"\nCTU-UHB before SMOTE: {dict(zip(*np.unique(y_ctu, return_counts=True)))}")
print(f"CTU-UHB after  SMOTE: {dict(zip(*np.unique(y_ctu_bal, return_counts=True)))}")

# === Train/Val/Test splits ===
X_tr_u, X_te_u, y_tr_u, y_te_u = train_test_split(X_uci_bal, y_uci_bal, test_size=0.2, stratify=y_uci_bal, random_state=42)
X_tr_u, X_va_u, y_tr_u, y_va_u = train_test_split(X_tr_u,    y_tr_u,    test_size=0.15, stratify=y_tr_u, random_state=42)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_ctu_bal, y_ctu_bal, test_size=0.2, stratify=y_ctu_bal, random_state=42)

print(f"\nUCI CTG   — Train: {len(X_tr_u)} | Val: {len(X_va_u)} | Test: {len(X_te_u)}")
print(f"CTU-UHB   — Train: {len(X_tr_c)} | Test: {len(X_te_c)}")


In [ ]:
from sklearn.pipeline import Pipeline

# Train and evaluate multiple classical models
classical_models = {
    'Random Forest':    RandomForestClassifier(n_estimators=300, max_depth=15,
                                               class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':          xgb.XGBClassifier(n_estimators=300, max_depth=6,
                                          learning_rate=0.05, use_label_encoder=False,
                                          eval_metric='mlogloss', random_state=42,
                                          tree_method='hist'),
    'LightGBM':         lgb.LGBMClassifier(n_estimators=500, num_leaves=63,
                                            learning_rate=0.03, class_weight='balanced',
                                            random_state=42, verbose=-1, n_jobs=-1),
    'CatBoost':         cb.CatBoostClassifier(iterations=300, learning_rate=0.05,
                                              depth=8, random_seed=42, verbose=0,
                                              auto_class_weights='Balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                     learning_rate=0.05, random_state=42),
}

classical_results = {}
for name, model in classical_models.items():
    print(f"Training {name}...")
    model.fit(X_tr_u, y_tr_u)
    y_pred = model.predict(X_te_u)
    y_proba = model.predict_proba(X_te_u)

    results_dict = {
        'Balanced Acc':   balanced_accuracy_score(y_te_u, y_pred),
        'Macro F1':       f1_score(y_te_u, y_pred, average='macro'),
        'Weighted F1':    f1_score(y_te_u, y_pred, average='weighted'),
        'Macro AUROC':    roc_auc_score(y_te_u, y_proba, multi_class='ovr', average='macro'),
        'Cohen Kappa':    cohen_kappa_score(y_te_u, y_pred),
        'Pathological F1':f1_score(y_te_u, y_pred, average=None)[2] if len(np.unique(y_te_u)) >= 3 else 0,
    }
    classical_results[name] = results_dict
    print(f"  Balanced Acc: {results_dict['Balanced Acc']:.4f} | "
          f"Macro F1: {results_dict['Macro F1']:.4f} | "
          f"AUROC: {results_dict['Macro AUROC']:.4f}")

classical_df = pd.DataFrame(classical_results).T
print("\n=== Classical Model Comparison (UCI CTG) ===")
print(classical_df.round(4).to_string())


In [ ]:
class FetalHealthMLP(nn.Module):
    def __init__(self, input_dim, n_classes=3, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout / 2),

            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, n_classes)
        )
        # Residual projection
        self.shortcut = nn.Linear(input_dim, 256)

    def forward(self, x):
        h1 = self.net[:6](x) + self.shortcut(x)  # residual
        return self.net[6:](h1)

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# Class weights for loss
class_weights_uci = compute_class_weight('balanced', classes=np.unique(y_tr_u), y=y_tr_u)
class_weights_tensor = torch.FloatTensor(class_weights_uci).to(device)

train_dl = DataLoader(TabularDataset(X_tr_u, y_tr_u), batch_size=128, shuffle=True)
val_dl   = DataLoader(TabularDataset(X_va_u, y_va_u), batch_size=128)
test_dl  = DataLoader(TabularDataset(X_te_u, y_te_u), batch_size=128)

mlp_model = FetalHealthMLP(X_tr_u.shape[1], n_classes=3).to(device)
optimizer = torch.optim.AdamW(mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

mlp_history = {'train_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0

for epoch in range(150):
    mlp_model.train()
    train_losses = []
    for X_b, y_b in train_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp_model(X_b), y_b)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    mlp_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_dl:
            preds = mlp_model(X_b.to(device)).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_b.numpy())
    val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    mlp_history['train_loss'].append(np.mean(train_losses))
    mlp_history['val_f1'].append(val_f1)
    scheduler.step()

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(mlp_model.state_dict(), 'best_mlp_fetal.pt')

mlp_model.load_state_dict(torch.load('best_mlp_fetal.pt'))
print(f"MLP best val F1: {best_val_f1:.4f}")


In [ ]:
# KANs replace MLPs with learnable activation functions on edges
# Outperform traditional ML on CTG achieving 93.6% accuracy (2025 paper)
try:
    from kan import KAN

    kan_model_fetal = KAN(
        width=[X_tr_u.shape[1], 64, 32, 3],
        grid=5,
        k=3,
        seed=42,
        device=device
    )

    # KAN uses its own training loop
    dataset_kan = {
        'train_input': torch.FloatTensor(X_tr_u).to(device),
        'train_label': torch.LongTensor(y_tr_u).to(device),
        'test_input':  torch.FloatTensor(X_te_u).to(device),
        'test_label':  torch.LongTensor(y_te_u).to(device),
    }

    def acc_fn(pred, label):
        return (pred.argmax(1) == label).float().mean()

    kan_results = kan_model_fetal.train(
        dataset_kan,
        opt='Adam',
        steps=100,
        lr=1e-3,
        lamb=0.001,
        metrics=(acc_fn,),
        loss_fn=nn.CrossEntropyLoss()
    )
    print("KAN training complete")
    KAN_AVAILABLE = True

except ImportError:
    print("KAN library not available — installing...")
    !pip install -q pykan
    KAN_AVAILABLE = False
    print("Restart runtime and re-run this cell after install")


In [ ]:
# Fixed-length FHR windows for CNN-BiLSTM
WINDOW_SEC = 600   # 10-minute windows
STEP_SEC   = 300   # 5-minute step (50% overlap)
FS_CTU     = 4     # 4 Hz

def create_fhr_windows(records, window_sec=600, step_sec=300, fs=4):
    """Sliding window segmentation of FHR signals."""
    X, y = [], []
    win_len  = window_sec * fs
    step_len = step_sec * fs

    for rec in records:
        fhr = rec['fhr']
        label = rec['label']
        # Normalize per-recording
        fhr_valid = fhr[fhr > 50]
        if len(fhr_valid) < win_len:
            continue
        mean_fhr = fhr_valid.mean()
        std_fhr  = fhr_valid.std() + 1e-8

        for start in range(0, len(fhr) - win_len, step_len):
            window = fhr[start:start + win_len]
            if (window > 50).mean() < 0.7:  # skip windows with >30% artifacts
                continue
            window_norm = (window - mean_fhr) / std_fhr
            X.append(window_norm)
            y.append(label)

    return np.array(X), np.array(y)

X_wave, y_wave = create_fhr_windows(ctu_records, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
print(f"Waveform windows: {X_wave.shape}  Labels: {np.unique(y_wave, return_counts=True)}")

X_wave_tr, X_wave_te, y_wave_tr, y_wave_te = train_test_split(
    X_wave, y_wave, test_size=0.2, stratify=y_wave, random_state=42)

class FHRWaveformDataset(Dataset):
    def __init__(self, X, y):
        # (N, L) → (N, 1, L) for Conv1d
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

wave_class_w = compute_class_weight('balanced', classes=np.unique(y_wave_tr), y=y_wave_tr)
wave_crit    = nn.CrossEntropyLoss(weight=torch.FloatTensor(wave_class_w).to(device))

wave_tr_dl = DataLoader(FHRWaveformDataset(X_wave_tr, y_wave_tr), batch_size=32, shuffle=True)
wave_te_dl = DataLoader(FHRWaveformDataset(X_wave_te, y_wave_te), batch_size=32)

class FHR_CNN_BiLSTM(nn.Module):
    def __init__(self, n_classes=2, dropout=0.3):
        super().__init__()
        # Multi-scale CNN front end
        self.conv_small  = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=8,  padding=4), nn.ReLU(), nn.MaxPool1d(4))
        self.conv_medium = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=32, padding=16), nn.ReLU(), nn.MaxPool1d(4))
        self.conv_large  = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=64, padding=32), nn.ReLU(), nn.MaxPool1d(4))

        self.merge_conv = nn.Sequential(
            nn.Conv1d(96, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )
        lstm_in = 128
        self.bilstm = nn.LSTM(lstm_in, 64, num_layers=2,
                               batch_first=True, bidirectional=True, dropout=dropout)
        self.attn = nn.Sequential(
            nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1), nn.Softmax(dim=1))
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_classes))

    def forward(self, x):
        # x: (B, 1, L)
        s = self.conv_small(x)   # (B, 32, L//4)
        m = self.conv_medium(x)
        lg = self.conv_large(x)
        # Align lengths
        min_len = min(s.size(2), m.size(2), lg.size(2))
        merged = torch.cat([s[:,:,:min_len], m[:,:,:min_len], lg[:,:,:min_len]], dim=1)
        feat = self.merge_conv(merged)              # (B, 128, L')
        feat = feat.permute(0, 2, 1)               # (B, L', 128)
        lstm_out, _ = self.bilstm(feat)             # (B, L', 128)
        attn_w = self.attn(lstm_out)
        context = (attn_w * lstm_out).sum(dim=1)   # (B, 128)
        return self.classifier(context)

wave_model = FHR_CNN_BiLSTM(n_classes=2).to(device)
print(f"CNN-BiLSTM params: {sum(p.numel() for p in wave_model.parameters()):,}")

wave_optimizer = torch.optim.AdamW(wave_model.parameters(), lr=1e-3, weight_decay=1e-4)
wave_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(wave_optimizer, T_max=50)
wave_history   = {'train_loss': [], 'val_f1': []}
best_wave_f1   = 0

for epoch in range(60):
    wave_model.train()
    losses = []
    for X_b, y_b in wave_tr_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        wave_optimizer.zero_grad()
        loss = wave_crit(wave_model(X_b), y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(wave_model.parameters(), 1.0)
        wave_optimizer.step()
        losses.append(loss.item())

    wave_model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for X_b, y_b in wave_te_dl:
            p = wave_model(X_b.to(device)).argmax(1).cpu().numpy()
            all_p.extend(p); all_l.extend(y_b.numpy())
    f1_val = f1_score(all_l, all_p, average='macro', zero_division=0)
    wave_history['train_loss'].append(np.mean(losses))
    wave_history['val_f1'].append(f1_val)
    wave_scheduler.step()

    if f1_val > best_wave_f1:
        best_wave_f1 = f1_val
        torch.save(wave_model.state_dict(), 'best_wave_fetal.pt')

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {np.mean(losses):.4f} | F1: {f1_val:.4f}")

wave_model.load_state_dict(torch.load('best_wave_fetal.pt'))
print(f"\nBest CNN-BiLSTM F1: {best_wave_f1:.4f}")


In [ ]:
def evaluate_fetal_model(model_type, model, X_test, y_test, label_names_dict, loader=None):
    if model_type == 'sklearn':
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
    else:
        model.eval()
        all_preds, all_probas = [], []
        with torch.no_grad():
            for X_b, y_b in loader:
                out = model(X_b.to(device))
                proba = F.softmax(out, dim=1).cpu().numpy()
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_probas.extend(proba)
        y_pred  = np.array(all_preds)
        y_proba = np.array(all_probas)

    n_classes = len(np.unique(y_test))
    metrics = {
        'Balanced Acc':     balanced_accuracy_score(y_test, y_pred),
        'Macro F1':         f1_score(y_test, y_pred, average='macro', zero_division=0),
        'Weighted F1':      f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'Cohen Kappa':      cohen_kappa_score(y_test, y_pred),
        'Macro AUROC':      roc_auc_score(y_test, y_proba,
                                          multi_class='ovr', average='macro')
                            if n_classes > 1 else 0,
    }
    # Per-class sensitivity/specificity
    for i, cls_name in label_names_dict.items():
        tp = ((y_pred == i) & (y_test == i)).sum()
        fn = ((y_pred != i) & (y_test == i)).sum()
        fp = ((y_pred == i) & (y_test != i)).sum()
        tn = ((y_pred != i) & (y_test != i)).sum()
        metrics[f'{cls_name}_Sens'] = tp / (tp + fn + 1e-8)
        metrics[f'{cls_name}_Spec'] = tn / (tn + fp + 1e-8)

    return metrics, y_pred, y_proba

# Best classical model
best_classical = max(classical_results, key=lambda k: classical_results[k]['Macro AUROC'])
best_clf = classical_models[best_classical]

metrics_clf, pred_clf, proba_clf = evaluate_fetal_model(
    'sklearn', best_clf, X_te_u, y_te_u, label_names)
metrics_mlp, pred_mlp, proba_mlp = evaluate_fetal_model(
    'torch', mlp_model, None, y_te_u, label_names, test_dl)
metrics_wave, pred_wave, proba_wave = evaluate_fetal_model(
    'torch', wave_model, None, y_wave_te, {0:'Normal', 1:'Acidosis'}, wave_te_dl)

print(f"\n{'='*60}")
print(f"Best Classical ({best_classical}) on UCI CTG:")
for k, v in metrics_clf.items(): print(f"  {k:30s}: {v:.4f}")
print(f"\nMLP on UCI CTG:")
for k, v in metrics_mlp.items(): print(f"  {k:30s}: {v:.4f}")
print(f"\nCNN-BiLSTM on CTU-UHB Raw Waveforms:")
for k, v in metrics_wave.items(): print(f"  {k:30s}: {v:.4f}")


In [ ]:
print("Computing SHAP values for best classical model...")
best_clf_for_shap = classical_models[best_classical]

# Use TreeExplainer for tree-based models
explainer = shap.TreeExplainer(best_clf_for_shap)
shap_values = explainer.shap_values(X_te_u[:200])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# SHAP summary plot (beeswarm)
plt.sca(axes[0])
shap.summary_plot(
    shap_values[2] if isinstance(shap_values, list) else shap_values,
    X_te_u[:200],
    feature_names=uci_feature_cols_avail,
    plot_type='dot',
    max_display=15,
    show=False,
    color_bar=True
)
axes[0].set_title('SHAP Feature Importance\n(Pathological class)',
                   fontsize=13, fontweight='bold')

# SHAP bar plot — mean absolute
shap_mean = np.abs(
    shap_values[2] if isinstance(shap_values, list) else shap_values
).mean(axis=0)
sorted_idx = np.argsort(shap_mean)[-15:]
feat_names = [uci_feature_cols_avail[i] for i in sorted_idx]

axes[1].barh(feat_names, shap_mean[sorted_idx],
             color='#e74c3c', edgecolor='black', alpha=0.85)
axes[1].set_title('Mean |SHAP| — Top 15 Features', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('shap_fetal_health.png', dpi=150, bbox_inches='tight')
plt.show()

# Map to clinical Dawes-Redman interpretation
print("\n=== SHAP → Clinical Mapping ===")
dr_map = {
    'LB': 'FHR Baseline (normal: 110-160 bpm)',
    'MSTV': 'Short-term variability (normal: >3ms)',
    'AC': 'Accelerations (normal: ≥2 per 20 min)',
    'ASTV': '% abnormal STV (concern: >20%)',
    'DP': 'Prolonged decelerations (concern: any)',
    'DS': 'Severe decelerations (concern: any)',
    'ALTV': '% abnormal LTV',
    'MLTV': 'Long-term variability',
}
top_feats = [uci_feature_cols_avail[i] for i in sorted_idx[-8:]][::-1]
for f in top_feats:
    if f in dr_map:
        print(f"  {f:8s}: {dr_map[f]}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

model_results = [
    (best_classical, pred_clf, proba_clf, y_te_u, label_names),
    ('MLP', pred_mlp, proba_mlp, y_te_u, label_names),
]

for col_idx, (model_name, y_pred, y_proba, y_true, lnames) in enumerate(model_results[:2]):
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=[lnames[i] for i in sorted(lnames)])
    disp.plot(ax=axes[0, col_idx], cmap='Blues', colorbar=False)
    axes[0, col_idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')

    # ROC per class
    for i, cls_name in lnames.items():
        from sklearn.preprocessing import label_binarize
        y_bin = label_binarize(y_true, classes=sorted(lnames.keys()))
        if y_bin.shape[1] > i:
            fpr, tpr, _ = __import__('sklearn').metrics.roc_curve(y_bin[:, i], y_proba[:, i])
            auc = roc_auc_score(y_bin[:, i], y_proba[:, i])
            axes[1, col_idx].plot(fpr, tpr, linewidth=1.5,
                                   label=f'{cls_name} (AUC={auc:.3f})')
    axes[1, col_idx].plot([0,1],[0,1],'k--', linewidth=0.8)
    axes[1, col_idx].set_title(f'{model_name} — ROC Curves', fontweight='bold')
    axes[1, col_idx].set_xlabel('FPR'); axes[1, col_idx].set_ylabel('TPR')
    axes[1, col_idx].legend(fontsize=9)
    axes[1, col_idx].grid(True, alpha=0.3)

# CNN-BiLSTM waveform confusion matrix
cm_wave = confusion_matrix(y_wave_te, pred_wave)
ConfusionMatrixDisplay(cm_wave, display_labels=['Normal', 'Acidosis']).plot(
    ax=axes[0, 2], cmap='Blues', colorbar=False)
axes[0, 2].set_title('CNN-BiLSTM (CTU-UHB)\nConfusion Matrix', fontweight='bold')

fpr_w, tpr_w, _ = __import__('sklearn').metrics.roc_curve(y_wave_te, proba_wave[:, 1])
auc_w = roc_auc_score(y_wave_te, proba_wave[:, 1])
axes[1, 2].plot(fpr_w, tpr_w, linewidth=2, color='#2ecc71', label=f'Acidosis AUC={auc_w:.3f}')
axes[1, 2].plot([0,1],[0,1],'k--'); axes[1, 2].legend(); axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].set_title('CNN-BiLSTM — ROC', fontweight='bold')

plt.suptitle('Fetal Health Analyser — Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_fetal_health.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import pickle

# Save best classical model (UCI CTG — 3-class)
with open('medverse_fetal_ctg_classifier.pkl', 'wb') as f:
    pickle.dump({
        'model': best_clf,
        'scaler': scaler_uci,
        'feature_names': uci_feature_cols_avail,
        'classes': {0: 'Normal', 1: 'Suspect', 2: 'Pathological'},
        'model_name': best_classical,
        'metrics': classical_results[best_classical],
    }, f)

# Save waveform model (CTU-UHB — binary acidosis)
torch.save({
    'model_state_dict': wave_model.state_dict(),
    'model_name': 'CNN-BiLSTM',
    'task': 'fetal_acidosis_binary',
    'sampling_rate': 4,
    'window_sec': WINDOW_SEC,
    'classes': {0: 'Normal', 1: 'Acidosis (pH<7.15)'},
    'best_f1': best_wave_f1,
    'input_shape': f'(1, {WINDOW_SEC * FS_CTU})',
    'trained_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}, 'medverse_fetal_waveform_model.pt')

with open('medverse_fetal_config.json', 'w') as f:
    json.dump({
        'models': {
            'tabular': 'medverse_fetal_ctg_classifier.pkl',
            'waveform': 'medverse_fetal_waveform_model.pt'
        },
        'tabular_features': uci_feature_cols_avail,
        'waveform_input': f'FHR signal, {FS_CTU}Hz, {WINDOW_SEC}s window',
        'classes_3': {'0': 'Normal', '1': 'Suspect', '2': 'Pathological'},
        'classes_binary': {'0': 'Normal', '1': 'Acidosis (pH<7.15)'},
        'dawes_redman_thresholds': {
            'baseline_fhr_min': 110, 'baseline_fhr_max': 160,
            'stv_min_ms': 3, 'min_accelerations': 2,
            'max_prolonged_decels': 0
        },
        'severity_mapping': {
            'Normal': 'normal', 'Suspect': 'watch', 'Pathological': 'critical'
        }
    }, f, indent=2)

print("Saved:")
print("  medverse_fetal_ctg_classifier.pkl  → tabular CTG (matches ctg_dawes_redman.py features)")
print("  medverse_fetal_waveform_model.pt   → raw FHR CNN-BiLSTM")
print("  medverse_fetal_config.json")


In [ ]:
def predict_fetal_health(fhr_array, uc_array=None, mode='waveform'):
    """
    Exactly matches what MedVerse AbdomenMonitor + ctg_dawes_redman.py sends.

    mode='tabular'  → uses UCI CTG features (computed from 30-min ring buffer)
    mode='waveform' → uses raw FHR signal from piezo/MEMS AbdomenMonitor
    """
    if mode == 'tabular':
        # Extract features matching UCI CTG schema
        feats = extract_fhr_features(fhr_array, uc_array)
        if not feats:
            return {'error': 'Insufficient signal quality'}

        # Map to UCI feature vector order
        feat_vec = np.array([feats.get(f.lower(), 0) for f in uci_feature_cols_avail]).reshape(1, -1)
        feat_scaled = scaler_uci.transform(feat_vec)
        pred_class = best_clf.predict(feat_scaled)[0]
        pred_proba = best_clf.predict_proba(feat_scaled)[0]

        return {
            'classification':  label_names[pred_class],
            'confidence':       float(pred_proba.max()),
            'probabilities':   {label_names[i]: float(p) for i, p in enumerate(pred_proba)},
            'baseline_fhr':    round(feats.get('baseline_fhr', 0), 1),
            'stv_ms':          round(feats.get('stv_ms', 0), 2),
            'n_accelerations': int(feats.get('n_accelerations', 0)),
            'n_decelerations': int(feats.get('n_decelerations', 0)),
            'reactive':        bool(feats.get('reactive', 0)),
            'dawes_redman_met': (
                110 <= feats.get('baseline_fhr', 0) <= 160 and
                feats.get('stv_ms', 0) >= 3 and
                feats.get('n_accelerations', 0) >= 1 and
                feats.get('n_decelerations', 0) == 0
            ),
            'severity': ['normal', 'watch', 'critical'][pred_class],
            'mode': 'tabular'
        }

    else:  # waveform mode
        win_len = WINDOW_SEC * FS_CTU
        if len(fhr_array) < win_len:
            pad = np.full(win_len - len(fhr_array), fhr_array.mean())
            fhr_array = np.concatenate([fhr_array, pad])
        window = fhr_array[-win_len:]
        window = (window - window.mean()) / (window.std() + 1e-8)
        x = torch.FloatTensor(window).unsqueeze(0).unsqueeze(0).to(device)

        wave_model.eval()
        with torch.no_grad():
            out = F.softmax(wave_model(x), dim=1).cpu().numpy()[0]

        return {
            'classification':  'Normal' if out.argmax() == 0 else 'Acidosis (pH<7.15)',
            'probabilities':   {'Normal': float(out[0]), 'Acidosis': float(out[1])},
            'confidence':      float(out.max()),
            'severity':        'normal' if out.argmax() == 0 else 'critical',
            'mode': 'waveform'
        }

# Test with a real CTU-UHB sample
test_rec = ctu_records[0]
result_wave = predict_fetal_health(test_rec['fhr'], test_rec['uc'], mode='waveform')
result_tab  = predict_fetal_health(test_rec['fhr'], test_rec['uc'], mode='tabular')

print("=== MedVerse Inference — Waveform Mode ===")
for k, v in result_wave.items(): print(f"  {k}: {v}")
print("\n=== MedVerse Inference — Tabular Mode ===")
for k, v in result_tab.items(): print(f"  {k}: {v}")
